# Investor Pitch Notebook

This notebook presents a decision‑ready view of the system for an investor:

- High‑level portfolio performance and risk
- Which sectors contributed most and **why** they perform well
- Which rules are most stable and profitable
- A **rule detail lookup** tool to inspect any rule in depth

All analysis is based on the research artifacts generated by the engine:

- `portfolio_equity_curve.csv`
- `portfolio_used_trades.csv`
- `rule_stability.csv`
- `sector_investability.csv`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

RESEARCH_DIR = "research"  # adjust if needed

def _load_csv(name: str) -> pd.DataFrame:
    path = os.path.join(RESEARCH_DIR, name)
    if not os.path.exists(path):
        print(f"[WARN] Missing research file: {name}")
        return pd.DataFrame()
    return pd.read_csv(path)

def load_research_artifacts():
    # 1) Equity curve
    equity = _load_csv("portfolio_equity_curve.csv")

    if not equity.empty:
        if "date" in equity.columns:
            equity["date"] = pd.to_datetime(equity["date"], errors="coerce")
            equity = equity.set_index("date").sort_index()
        else:
            first = equity.columns[0]
            equity[first] = pd.to_datetime(equity[first], errors="coerce")
            equity = equity.set_index(first).sort_index()

        if "equity" in equity.columns:
            equity["equity"] = pd.to_numeric(equity["equity"], errors="coerce")
        else:
            num_cols = equity.select_dtypes(include=["number"]).columns
            if len(num_cols) > 0:
                equity["equity"] = pd.to_numeric(equity[num_cols[0]], errors="coerce")

    # 2) Used trades
    used_trades = _load_csv("portfolio_used_trades.csv")
    if not used_trades.empty:
        for col in ["entry_date", "exit_date", "signal_date"]:
            if col in used_trades.columns:
                used_trades[col] = pd.to_datetime(used_trades[col], errors="coerce")
        if "ret" in used_trades.columns:
            used_trades["ret"] = pd.to_numeric(used_trades["ret"], errors="coerce")

    # 3) Rule stability
    stability = _load_csv("rule_stability.csv")

    # 4) Sector investability
    sectors = _load_csv("sector_investability.csv")

    return equity, used_trades, stability, sectors

def compute_basic_metrics(equity: pd.DataFrame) -> dict:
    if equity.empty:
        return {}

    if "equity" in equity.columns:
        eq = equity["equity"].astype(float)
    else:
        num_cols = equity.select_dtypes(include=["number"]).columns
        if len(num_cols) == 0:
            return {}
        eq = equity[num_cols[0]].astype(float)

    if not isinstance(eq.index, pd.DatetimeIndex):
        eq.index = pd.to_datetime(eq.index, errors="coerce")

    eq = eq.dropna()
    if eq.empty:
        return {}

    ret = eq.pct_change().dropna()

    total_return = eq.iloc[-1] / eq.iloc[0] - 1.0
    ann_factor = 252
    ann_return = (1 + total_return) ** (ann_factor / len(eq)) - 1.0 if len(eq) > 1 else np.nan
    ann_vol = ret.std() * np.sqrt(ann_factor)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    max_dd = ((eq / eq.cummax()) - 1.0).min()

    return {
        "start": eq.index.min(),
        "end": eq.index.max(),
        "total_return": total_return,
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "series": eq,
        "returns": ret,
    }

def plot_drawdown(eq: pd.Series):
    if eq.empty:
        return
    running_max = eq.cummax()
    dd = (eq / running_max) - 1.0

    plt.figure(figsize=(10, 4))
    dd.plot()
    plt.title("Drawdown (Underwater) Curve")
    plt.ylabel("Drawdown")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

equity, used_trades, stability, sectors = load_research_artifacts()
metrics = compute_basic_metrics(equity)
metrics

## 1. Investor Overview

High‑level performance and risk profile of the portfolio.

In [ ]:
if not metrics:
    print("No equity data available.")
else:
    print("=== INVESTOR OVERVIEW ===\n")
    print(f"Equity curve from {metrics['start'].date()} to {metrics['end'].date()}")
    print(f"Total return: {metrics['total_return']*100:.2f}%")
    print(f"Annualized return: {metrics['ann_return']*100:.2f}%")
    print(f"Annualized volatility: {metrics['ann_vol']*100:.2f}%")
    print(f"Sharpe ratio: {metrics['sharpe']:.2f}")
    print(f"Max drawdown: {metrics['max_drawdown']*100:.2f}%")

    eq = metrics["series"]
    ax = eq.plot(title="Portfolio Equity Curve", figsize=(10, 5))
    ax.set_ylabel("Equity")
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    plot_drawdown(eq)

## 2. What Drove Performance?

- Distribution of trade returns
- Top rules by cumulative PnL
- Narrative for an investor: selective rule inclusion, sector timing, and consistent capture of medium‑sized moves.

In [ ]:
print("=== WHAT DROVE PERFORMANCE? ===")

if not metrics:
    print("Insufficient data to analyze performance drivers.")
else:
    print(
        f"The portfolio delivered a total return of {metrics['total_return']*100:.2f}% "
        f"and an annualized return of {metrics['ann_return']*100:.2f}% with a Sharpe ratio of {metrics['sharpe']:.2f}."
    )
    print(
        "Performance was driven by selective rule inclusion, sector timing, and consistent capture "
        "of medium-sized moves rather than a few outsized winners."
    )

if used_trades.empty:
    print("\nNo trade data available.")
else:
    plt.figure(figsize=(8, 4))
    used_trades["ret"].hist(bins=40)
    plt.title("Distribution of Trade Returns")
    plt.xlabel("Return")
    plt.ylabel("Count")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    if "rule_id" in used_trades.columns:
        rule_pnl = used_trades.groupby("rule_id")["ret"].sum().sort_values(ascending=False)

        print("\nTop 10 rules by cumulative PnL:")
        display(rule_pnl.head(10).to_frame("cumulative_pnl"))

        plt.figure(figsize=(10, 4))
        rule_pnl.head(10).plot(kind="bar")
        plt.title("Top 10 Rules by Cumulative PnL")
        plt.ylabel("Cumulative Return")
        plt.grid(True, axis="y")
        plt.tight_layout()
        plt.show()

## 3. Which Sectors Contributed Most — and Why?

Here we connect **realized PnL** with **sector stability and investability**:

- Which sectors actually drove returns
- How their mean, win rate, Sharpe, and stability look
- Why top sectors (e.g., BASE_METALS, INDUSTRIALS_ELEC) are investable while others (e.g., MEDS_1) are not.

In [ ]:
print("=== WHICH SECTORS CONTRIBUTED MOST? ===")

if used_trades.empty:
    print("No trade data available.")
else:
    sector_pnl = used_trades.groupby("group")["ret"].sum().sort_values(ascending=False)
    sector_counts = used_trades.groupby("group")["ret"].count().sort_values(ascending=False)

    print("\nSector cumulative PnL:")
    display(sector_pnl.to_frame("pnl"))

    print("\nSector trade counts:")
    display(sector_counts.to_frame("trade_count"))

    plt.figure(figsize=(10, 4))
    sector_pnl.plot(kind="bar")
    plt.title("Cumulative PnL by Sector")
    plt.ylabel("Cumulative Return")
    plt.grid(True, axis="y")
    plt.tight_layout()
    plt.show()

if not sectors.empty:
    merged = sectors.set_index("group").join(sector_pnl.rename("pnl"), how="left")
    merged["pnl"] = merged["pnl"].fillna(0)

    cols = ["mean", "win_rate", "sharpe", "sortino", "stability", "is_investable", "pnl"]
    cols = [c for c in cols if c in merged.columns]

    print("\nSector stability and investability (with realized PnL):")
    display(merged[cols].sort_values("pnl", ascending=False))

## 4. Which Rules Were Most Stable?

We now focus on **rule‑level stability and quality**:

- Top rules by rule_quality_score
- Their long‑term and recent performance
- How that translates into realized PnL in the portfolio.

In [ ]:
print("=== WHICH RULES WERE MOST STABLE? ===")

if stability.empty:
    print("No rule stability data available.")
else:
    top_rules = stability.sort_values("rule_quality_score", ascending=False).head(10)

    cols = [
        "rule_id", "group", "n_trades", "avg_ret_full", "win_rate",
        "max_dd", "avg_ret_prev_90d", "avg_ret_prev_30d",
        "rule_quality_score", "is_investable"
    ]
    cols = [c for c in cols if c in top_rules.columns]

    print("\nTop 10 rules by stability / quality:")
    display(top_rules[cols])

    plt.figure(figsize=(10, 4))
    top_rules.set_index("rule_id")["rule_quality_score"].plot(kind="bar")
    plt.title("Top 10 Rules by Rule Quality Score")
    plt.ylabel("Rule Quality Score")
    plt.grid(True, axis="y")
    plt.tight_layout()
    plt.show()

    if not used_trades.empty:
        rule_pnl = used_trades.groupby("rule_id")["ret"].sum()
        merged_rules = top_rules.set_index("rule_id").join(rule_pnl.rename("pnl"), how="left")
        merged_rules["pnl"] = merged_rules["pnl"].fillna(0)

        cols2 = ["group", "n_trades", "avg_ret_full", "win_rate", "max_dd", "rule_quality_score", "pnl"]
        cols2 = [c for c in cols2 if c in merged_rules.columns]

        print("\nTop rules with realized PnL:")
        display(merged_rules[cols2].sort_values("pnl", ascending=False))

## 5. Rule Detail Lookup Tool

This section lets you look up any rule by `rule_id` and see:

- Its stability metrics
- Its realized PnL and trade distribution
- How it behaves over time

You can use this live in front of an investor to answer:

> *“Show me exactly how this rule behaves, how often it trades, and what its drawdowns look like.”*

In [ ]:
from ipywidgets import interact, Dropdown, fixed

def rule_detail_view(rule_id: str, stability: pd.DataFrame, used_trades: pd.DataFrame):
    if stability.empty:
        print("No stability data available.")
        return

    rule_row = stability[stability["rule_id"] == rule_id]
    if rule_row.empty:
        print(f"No stability record found for rule_id = {rule_id}.")
        return

    print("\n=== RULE STABILITY SNAPSHOT ===")
    display(rule_row)

    if used_trades.empty:
        print("No trade data available for this rule.")
        return

    rule_trades = used_trades[used_trades["rule_id"] == rule_id].copy()
    if rule_trades.empty:
        print("This rule did not participate in the portfolio backtest.")
        return

    rule_trades = rule_trades.sort_values("entry_date")
    rule_trades["pnl_pct"] = rule_trades["ret"] * 100

    print("\n=== RULE TRADE SUMMARY ===")
    print(f"Number of trades in portfolio: {len(rule_trades)}")
    print(f"Average trade return: {rule_trades['ret'].mean()*100:.2f}%")
    print(f"Win rate: {(rule_trades['ret'] > 0).mean()*100:.2f}%")
    print(f"Worst trade: {rule_trades['ret'].min()*100:.2f}%")
    print(f"Best trade: {rule_trades['ret'].max()*100:.2f}%")

    display(rule_trades[["group", "ticker", "entry_date", "exit_date", "ret"]].head(10))

    plt.figure(figsize=(8, 4))
    rule_trades["ret"].hist(bins=30)
    plt.title(f"Distribution of Trade Returns for {rule_id}")
    plt.xlabel("Return")
    plt.ylabel("Count")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Cumulative PnL over time
    rule_trades = rule_trades.sort_values("exit_date")
    rule_trades["cum_pnl"] = rule_trades["ret"].cumsum()

    plt.figure(figsize=(8, 4))
    plt.plot(rule_trades["exit_date"], rule_trades["cum_pnl"])
    plt.title(f"Cumulative PnL Over Time for {rule_id}")
    plt.xlabel("Exit Date")
    plt.ylabel("Cumulative Return")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

if stability.empty:
    print("No stability data available for rule lookup.")
else:
    rule_ids = stability["rule_id"].unique()
    interact(
        rule_detail_view,
        rule_id=Dropdown(options=sorted(rule_ids), description="Rule ID"),
        stability=fixed(stability),
        used_trades=fixed(used_trades),
    )

## 6. Risk Overview

Summarizes concentration and structural risks in the system:

- Sector concentration
- Rule concentration
- Drawdown profile
- Sectors excluded by strict stability filters
- Regime dependency

In [ ]:
print("=== WHAT RISKS REMAIN? ===")

if not used_trades.empty:
    sector_counts = used_trades["group"].value_counts(normalize=True)
    top_sector = sector_counts.index[0]
    print(
        f"- Sector concentration: {top_sector} accounts for {sector_counts.iloc[0]*100:.1f}% of all trades."
    )

    rule_counts = used_trades["rule_id"].value_counts(normalize=True)
    top_rule = rule_counts.index[0]
    print(
        f"- Rule concentration: rule {top_rule} represents {rule_counts.iloc[0]*100:.1f}% of all trades."
    )

if metrics:
    print(
        f"- Drawdown risk: historical max drawdown of {metrics['max_drawdown']*100:.2f}%."
    )

if not sectors.empty:
    weak = sectors.loc[~sectors["is_investable"], "group"].tolist()
    if weak:
        print(
            "- Several sectors are excluded due to weak historical behavior: "
            + ", ".join(weak[:10])
            + ("..." if len(weak) > 10 else "")
        )

print(
    "- Regime dependency: the system performs best when leader/lagger relationships are stable "
    "and volatility is moderate."
)